# COMP5318 Assignment 1: Rice Classification

##### Group number: 54
##### Student 1 SID: 550344487
##### Student 2 SID: 550894775
##### Student 3 SID: 550610795
##### Student 4 SID: ... 

## **1. Data Pre-processing**

In [2]:
# Import all libraries
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer # missing value
from sklearn.preprocessing import MinMaxScaler # normalisation
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import f1_score
import pandas as pd
import numpy as np

In [3]:
# Ignore future warnings
from warnings import simplefilter
simplefilter(action='ignore', category=FutureWarning)

In [4]:
# Load the rice dataset: rice-final2.csv
df = pd.read_csv('rice-final2.csv')
col = df.columns
class_col = col[-1]
df.head(5)


,Area,Perimiter,Major_Axis_Length,Minor_Axis_Length,Eccentricity,Convex_Area,Extent,class
0,12573,461.4660034,192.9033508,84.57207489,0.898771763,12893,0.550433397,class2
1,12845,464.1210022,194.3322144,85.52433777,0.897951961,13125,0.774962306,class2
2,14055,488.7489929,207.7517548,87.25032806,0.907536149,14484,0.550076306,class1
3,14412,490.3240051,207.4761353,89.68951416,0.901735425,14703,0.598853171,class1
4,14658,477.1170044,189.5666351,99.99777985,0.849550545,15048,0.649503708,class2


In [5]:
# check how many missing value for each col
col = df.columns
class_col = col[-1]

for i in col:
    n_missing = df[df[i] == "?"][i].count()
    print(f'Missing value count in {i}: {n_missing}')

Missing value count in Area: 4
Missing value count in Perimiter: 4
Missing value count in Major_Axis_Length: 5
Missing value count in Minor_Axis_Length: 3
Missing value count in Eccentricity: 6
Missing value count in Convex_Area: 5
Missing value count in Extent: 2
Missing value count in class: 0


In [6]:
# Pre-process dataset
# replace '?' with column mean using sklearn.impute.SimpleImputer
exe_df = df.copy()
imputer = SimpleImputer(strategy='mean')
col_class = exe_df[class_col]
# remove class column to prevent error
exe_df = exe_df.drop(columns=class_col)
exe_df = exe_df.replace('?', np.nan).astype(float)
imp_df = imputer.fit_transform(exe_df)  # return array
imp_df = pd.DataFrame(imp_df, columns=exe_df.columns, index=exe_df.index)
imp_df[class_col] = col_class

In [7]:
# check if any '?' in the data
col = imp_df.columns

for i in col:
    n_missing = imp_df[imp_df[i] == "?"][i].count()
    print(f'Missing value count in {i}: {n_missing}')

Missing value count in Area: 0
Missing value count in Perimiter: 0
Missing value count in Major_Axis_Length: 0
Missing value count in Minor_Axis_Length: 0
Missing value count in Eccentricity: 0
Missing value count in Convex_Area: 0
Missing value count in Extent: 0
Missing value count in class: 0


In [8]:
# normalisation
nor_df = imp_df.copy()
scaler = MinMaxScaler()
col_class = nor_df[class_col]
nor_df = nor_df.drop(columns=class_col)
nored_df = scaler.fit_transform(nor_df)
nored_df = pd.DataFrame(nored_df, columns=nor_df.columns, index=nor_df.index)
for i in nored_df.columns:
    print(f'{i} - min_value: {nored_df[i].min()}, max_value: {nored_df[i].max()}')
nored_df[class_col] = col_class

Area - min_value: 0.0, max_value: 1.0
Perimiter - min_value: 0.0, max_value: 0.9999999999999998
Major_Axis_Length - min_value: 0.0, max_value: 1.0000000000000002
Minor_Axis_Length - min_value: 0.0, max_value: 1.0
Eccentricity - min_value: 0.0, max_value: 1.0
Convex_Area - min_value: 0.0, max_value: 1.0
Extent - min_value: 0.0, max_value: 1.0


In [9]:
# replace class
nored_df[class_col] = nored_df[class_col].map({'class1': 0, 'class2': 1})
nored_df[class_col] = nored_df[class_col].astype(int)
processed_df = nored_df.copy()

In [10]:
# Print first ten rows of pre-processed dataset to 4 decimal places as per assignment spec
# A function is provided to assist

def print_data(X, y, n_rows=10):
    """Takes a numpy data array and target and prints the first ten rows.
    
    Arguments:
        X: numpy array of shape (n_examples, n_features)
        y: numpy array of shape (n_examples)
        n_rows: numpy of rows to print
    """
    for example_num in range(n_rows):
        for feature in X[example_num]:
            print("{:.4f}".format(feature), end=",")

        if example_num == len(X)-1:
            print(y[example_num],end="")
        else:
            print(y[example_num])
# array of values without y (class)
x = processed_df.drop(columns=class_col).values
# array of y values
y = processed_df[class_col].values

print_data(x, y)

0.4628,0.5406,0.5113,0.4803,0.7380,0.4699,0.1196,1
0.4900,0.5547,0.5266,0.5018,0.7319,0.4926,0.8030,1
0.6109,0.6847,0.6707,0.5409,0.8032,0.6253,0.1185,0
0.6466,0.6930,0.6677,0.5961,0.7601,0.6467,0.2669,0
0.6712,0.6233,0.4755,0.8293,0.3721,0.6803,0.4211,1
0.2634,0.2932,0.2414,0.4127,0.5521,0.2752,0.2825,1
0.8175,0.9501,0.9515,0.5925,0.9245,0.8162,0.0000,0
0.3174,0.3588,0.3601,0.3908,0.6921,0.3261,0.8510,1
0.3130,0.3050,0.2150,0.5189,0.3974,0.3159,0.4570,1
0.5120,0.5237,0.4409,0.6235,0.5460,0.5111,0.3155,1


## **2. Build Classifiers**

- Part 1:  Logistic Regression, Naïve Bayes
- Part 2:  KNN, Decision Tree, Ada Boost, Gradient Boost, Random Forest, SVM

### Part 1: Cross-validation without parameter tuning

In [11]:
## Setting the 10 fold stratified cross-validation
cvKFold=StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

# The stratified folds from cvKFold should be provided to the classifiers

In [12]:
# Logistic Regression
logreg = LogisticRegression(random_state=0)
logreg_scores = cross_val_score(logreg, x, y, cv=cvKFold)
logreg_acc = logreg_scores.mean()

In [13]:
# Naïve Bayes
nb = GaussianNB()
nb_scores = cross_val_score(nb, x, y, cv=cvKFold)
nb_acc = nb_scores.mean()

### Part 1 Results


In [14]:
# Print results for each classifier in part 1 to 4 decimal places here:
print("LogR average cross-validation accuracy: {:.4f}".format(logreg_acc))
print("NB average cross-validation accuracy: {:.4f}".format(nb_acc))

LogR average cross-validation accuracy: 0.9386
NB average cross-validation accuracy: 0.9264


### Part 2: Cross-validation with parameter tuning

Pipeline (do not skip the split):
1. Split pre-processed `x`, `y` **once** with `train_test_split` (`stratify=y`, `random_state=0`).
2. Tune each classifier on **train only** with `GridSearchCV(..., cv=cvKFold)`.
3. Report best params, best CV accuracy (`best_score_`), and test accuracy.
4. Random Forest also reports test macro F1 and weighted F1.

`test_size` is not specified in the PDF; sklearn default is 0.25.

In [15]:
# Split once for all Part 2 classifiers (not per model).
# PDF: train_test_split with stratification and random_state=0.
# test_size omitted on purpose -> sklearn default 0.25.
X_train, X_test, y_train, y_test = train_test_split(
    x, y, stratify=y, random_state=0
)

In [16]:
# KNN
# parameters you may consider
k = [1, 3, 5, 7]
p = [1, 2]
knn_param_grid = {'n_neighbors': k, 'p': p}

knn_grid = GridSearchCV(KNeighborsClassifier(), knn_param_grid, cv=cvKFold)
knn_grid.fit(X_train, y_train)
knn_best_k = knn_grid.best_params_['n_neighbors']
knn_best_p = knn_grid.best_params_['p']
knn_cv_acc = knn_grid.best_score_
knn_test_acc = knn_grid.score(X_test, y_test)

In [17]:
# Decision Tree
# parameters you may consider
max_depth = [3, 5, 7, 10]
min_samples_split = [2, 5, 10]
min_samples_leaf = [1, 2, 4]
dt_param_grid = {
    'max_depth': max_depth,
    'min_samples_split': min_samples_split,
    'min_samples_leaf': min_samples_leaf,
}

dt_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=0), dt_param_grid, cv=cvKFold
)
dt_grid.fit(X_train, y_train)
dt_best_max_depth = dt_grid.best_params_['max_depth']
dt_best_min_samples_split = dt_grid.best_params_['min_samples_split']
dt_best_min_samples_leaf = dt_grid.best_params_['min_samples_leaf']
dt_cv_acc = dt_grid.best_score_
dt_test_acc = dt_grid.score(X_test, y_test)

In [18]:
# Ada Boost
# parameters you may consider
n_estimators = [50, 100, 150]
learning_rate = [0.1, 0.2, 0.3, 0.5]
ada_param_grid = {
    'n_estimators': n_estimators,
    'learning_rate': learning_rate,
}

ada_grid = GridSearchCV(
    AdaBoostClassifier(random_state=0), ada_param_grid, cv=cvKFold
)
ada_grid.fit(X_train, y_train)
ada_best_n_estimators = ada_grid.best_params_['n_estimators']
ada_best_learning_rate = ada_grid.best_params_['learning_rate']
ada_cv_acc = ada_grid.best_score_
ada_test_acc = ada_grid.score(X_test, y_test)

In [19]:
# Gradient Boost
# parameters you may consider
max_depth = [1, 3, 5, 7]
n_estimators = [50, 100, 150]
learning_rate = [0.1, 0.2, 0.3, 0.5]
gb_param_grid = {
    'max_depth': max_depth,
    'n_estimators': n_estimators,
    'learning_rate': learning_rate,
}

gb_grid = GridSearchCV(
    GradientBoostingClassifier(random_state=0), gb_param_grid, cv=cvKFold
)
gb_grid.fit(X_train, y_train)
gb_best_max_depth = gb_grid.best_params_['max_depth']
gb_best_n_estimators = gb_grid.best_params_['n_estimators']
gb_best_learning_rate = gb_grid.best_params_['learning_rate']
gb_cv_acc = gb_grid.best_score_
gb_test_acc = gb_grid.score(X_test, y_test)

In [20]:
# Random Forest
# RandomForestClassifier: information gain (criterion='entropy') and max_features='sqrt'
# parameters you may consider
n_estimators = [10, 30, 60, 100]
max_leaf_nodes = [6, 12]
rf_param_grid = {
    'n_estimators': n_estimators,
    'max_leaf_nodes': max_leaf_nodes,
}

rf_grid = GridSearchCV(
    RandomForestClassifier(
        criterion='entropy', max_features='sqrt', random_state=0
    ),
    rf_param_grid,
    cv=cvKFold,
)
rf_grid.fit(X_train, y_train)
rf_best_n_estimators = rf_grid.best_params_['n_estimators']
rf_best_max_leaf_nodes = rf_grid.best_params_['max_leaf_nodes']
rf_cv_acc = rf_grid.best_score_
rf_test_acc = rf_grid.score(X_test, y_test)
rf_pred = rf_grid.predict(X_test)
rf_f1_macro = f1_score(y_test, rf_pred, average='macro')
rf_f1_weighted = f1_score(y_test, rf_pred, average='weighted')

In [21]:
# SVM
# parameters you may consider
C = [0.01, 0.1, 1, 5]
gamma = [0.01, 0.1, 1, 10]
# optional — leave empty to use SVC default kernel 'rbf'
kernel = []
svm_param_grid = {'C': C, 'gamma': gamma}

svm_grid = GridSearchCV(SVC(random_state=0), svm_param_grid, cv=cvKFold)
svm_grid.fit(X_train, y_train)
svm_best_C = svm_grid.best_params_['C']
svm_best_gamma = svm_grid.best_params_['gamma']
svm_cv_acc = svm_grid.best_score_
svm_test_acc = svm_grid.score(X_test, y_test)

### Part 2: Results

In [22]:
# Print results for each classifier here.
# Accuracy / F1 -> .4f ; k, p, n_estimators, max_leaf_nodes -> integers.

print("KNN best k: {}".format(knn_best_k))
print("KNN best p: {}".format(knn_best_p))
print("KNN cross-validation accuracy: {:.4f}".format(knn_cv_acc))
print("KNN test set accuracy: {:.4f}".format(knn_test_acc))

print("DT best max_depth: {}".format(dt_best_max_depth))
print("DT best min_samples_split: {}".format(dt_best_min_samples_split))
print("DT best min_samples_leaf: {}".format(dt_best_min_samples_leaf))
print("DT cross-validation accuracy: {:.4f}".format(dt_cv_acc))
print("DT test set accuracy: {:.4f}".format(dt_test_acc))

print("AdaBoost best n_estimators: {}".format(ada_best_n_estimators))
print("AdaBoost best learning_rate: {}".format(ada_best_learning_rate))
print("AdaBoost cross-validation accuracy: {:.4f}".format(ada_cv_acc))
print("AdaBoost test set accuracy: {:.4f}".format(ada_test_acc))

print("GB best max_depth: {}".format(gb_best_max_depth))
print("GB best n_estimators: {}".format(gb_best_n_estimators))
print("GB best learning_rate: {}".format(gb_best_learning_rate))
print("GB cross-validation accuracy: {:.4f}".format(gb_cv_acc))
print("GB test set accuracy: {:.4f}".format(gb_test_acc))

print("RF best n_estimators: {}".format(rf_best_n_estimators))
print("RF best max_leaf_nodes: {}".format(rf_best_max_leaf_nodes))
print("RF cross-validation accuracy: {:.4f}".format(rf_cv_acc))
print("RF test set accuracy: {:.4f}".format(rf_test_acc))
print("RF test set macro average F1: {:.4f}".format(rf_f1_macro))
print("RF test set weighted average F1: {:.4f}".format(rf_f1_weighted))

print("SVM best C: {}".format(svm_best_C))
print("SVM best gamma: {}".format(svm_best_gamma))
print("SVM cross-validation accuracy: {:.4f}".format(svm_cv_acc))
print("SVM test set accuracy: {:.4f}".format(svm_test_acc))

KNN best k: 5
KNN best p: 1
KNN cross-validation accuracy: 0.9371
KNN test set accuracy: 0.9257
DT best max_depth: 5
DT best min_samples_split: 5
DT best min_samples_leaf: 1
DT cross-validation accuracy: 0.9314
DT test set accuracy: 0.9400
AdaBoost best n_estimators: 150
AdaBoost best learning_rate: 0.5
AdaBoost cross-validation accuracy: 0.9438
AdaBoost test set accuracy: 0.9429
GB best max_depth: 1
GB best n_estimators: 50
GB best learning_rate: 0.3
GB cross-validation accuracy: 0.9448
GB test set accuracy: 0.9457
RF best n_estimators: 30
RF best max_leaf_nodes: 12
RF cross-validation accuracy: 0.9390
RF test set accuracy: 0.9371
RF test set macro average F1: 0.9355
RF test set weighted average F1: 0.9370
SVM best C: 5
SVM best gamma: 1
SVM cross-validation accuracy: 0.9457
SVM test set accuracy: 0.9343


### Test your code

The pipeline was verified on `test-before.csv` (209 rows, 6 features) and ran end to end
without errors, which checks that nothing is hard-coded to the rice column names or shape.

The code below is commented out because the submitted notebook must report results for the
rice dataset only. Uncomment it to re-run the runnability check on another dataset.

In [23]:
# Load the test dataset to test out your model.
# Commented out so the submitted notebook reports rice results only.
# Change the file name below to run the same pipeline on any other dataset.

# TEST_FILE = 'test-before.csv'
#
# test_df = pd.read_csv(TEST_FILE)
# test_class_col = test_df.columns[-1]
#
# test_X_df = test_df.drop(columns=test_class_col).replace('?', np.nan).astype(float)
# test_X_imp = SimpleImputer(strategy='mean').fit_transform(test_X_df)
# test_X = MinMaxScaler().fit_transform(test_X_imp)
# test_y = test_df[test_class_col].map({'class1': 0, 'class2': 1}).astype(int).values
#
# print("test shape: {}".format(test_X.shape))
# print_data(test_X, test_y, n_rows=5)
#
# # Part 1: cross-validation without parameter tuning
# print("TEST LogR: {:.4f}".format(
#     cross_val_score(LogisticRegression(random_state=0), test_X, test_y, cv=cvKFold).mean()))
# print("TEST NB: {:.4f}".format(
#     cross_val_score(GaussianNB(), test_X, test_y, cv=cvKFold).mean()))
#
# # Part 2: split once, then grid search on the training set only
# test_X_train, test_X_test, test_y_train, test_y_test = train_test_split(
#     test_X, test_y, stratify=test_y, random_state=0
# )
#
# test_estimators = {
#     'KNN': (KNeighborsClassifier(), knn_param_grid),
#     'DT': (DecisionTreeClassifier(random_state=0), dt_param_grid),
#     'AdaBoost': (AdaBoostClassifier(random_state=0), ada_param_grid),
#     'GB': (GradientBoostingClassifier(random_state=0), gb_param_grid),
#     'RF': (RandomForestClassifier(
#         criterion='entropy', max_features='sqrt', random_state=0), rf_param_grid),
#     'SVM': (SVC(random_state=0), svm_param_grid),
# }
#
# for name, (estimator, param_grid) in test_estimators.items():
#     grid = GridSearchCV(estimator, param_grid, cv=cvKFold)
#     grid.fit(test_X_train, test_y_train)
#     print("TEST {} best params: {}".format(name, grid.best_params_))
#     print("TEST {} cross-validation accuracy: {:.4f}".format(name, grid.best_score_))
#     print("TEST {} test set accuracy: {:.4f}".format(
#         name, grid.score(test_X_test, test_y_test)))
#     if name == 'RF':
#         test_rf_pred = grid.predict(test_X_test)
#         print("TEST RF test set macro average F1: {:.4f}".format(
#             f1_score(test_y_test, test_rf_pred, average='macro')))
#         print("TEST RF test set weighted average F1: {:.4f}".format(
#             f1_score(test_y_test, test_rf_pred, average='weighted')))
#
# print("test pipeline finished without errors")

## **3. Reflection and Discussion**



#### Overall Performance: A Narrow, Practically Insignificant Spread

All eight classifiers sit in a narrow band (about 0.926–0.946). With a 350-example
test set, one extra correct prediction is only 0.29 percentage points, so most of
the gaps we see are not practically meaningful. Gradient Boosting had the highest
test accuracy (0.9457) and SVM the highest cross-validation accuracy (0.9457), but
Logistic Regression with **no** tuning already reached 0.9386 — on par with tuned
KNN, Decision Tree and Random Forest. The best Gradient Boosting model used
`max_depth=1` (decision stumps), which is another sign that this rice data is close
to linearly separable: extra tree depth does not buy much. In other words, the
ceiling on this dataset appears to be set by the features themselves rather than by
model choice or tuning effort.

#### Naive Bayes Trails Due to Violated Independence Assumptions

Naive Bayes was the weakest performer (0.9264). GaussianNB assumes independent
features, but several morphological measurements here are almost the same quantity
measured twice: Area and Convex_Area correlate at 0.995, and Perimeter with
Major_Axis_Length at 0.972. The independence assumption is clearly false for this
dataset. The fact that NB still stays above 0.92 despite this is consistent with it
being fairly robust to that kind of violation in practice, but the correlated
features are the right explanation for why it trails the other seven classifiers
rather than any deeper deficiency in the algorithm itself.

#### Hyperparameter Tuning Helps Unevenly Across Models

Tuning helped different classifiers by very different amounts. On the same
train/test split, moving from sklearn defaults to the grid-search models changed
test accuracy as follows: Decision Tree +0.0171, Gradient Boosting +0.0143, KNN and
Random Forest +0.0057, SVM +0.0029, AdaBoost +0.0000. The largest gain is for the
Decision Tree: an unconstrained tree can grow until it overfits, and the selected
`max_depth=5` acts as regularisation, directly correcting that failure mode. Ensemble
methods already reduce variance on their own, so searching `n_estimators` and
`learning_rate` on this data mostly confirms a setting that was already good rather
than discovering a meaningfully better one. The takeaway is that tuning earns its
cost when the base model is prone to overfitting, not as a default upgrade that
should be expected to help every algorithm equally.

#### Cross-Validation and Test Rankings Disagree

Model rankings are not stable between cross-validation and the held-out test set.
SVM led on CV (0.9457) but dropped to 0.9343 on the test set; the Decision Tree did
the opposite (CV 0.9314, test 0.9400). Picking a "winning" model by its CV score
alone would therefore tend to inflate that winner's expected performance. Separately,
for Random Forest, macro F1 (0.9355) and weighted F1 (0.9370) come out almost
identical, because the classes are only mildly imbalanced (600 vs 800 examples).
Accuracy is a reasonable one-number summary here; it would not be if one class
dominated the dataset.

#### Methodological Caveats

**Caveat 1 — Data leakage (quantified).** The imputer and scaler were originally fit
on the full dataset before cross-validation and before the train/test split, which
leaks a small amount of test-fold information into preprocessing. To check how much
this actually mattered, we re-ran the full pipeline with a strictly no-leakage
procedure — imputer/scaler refit inside every CV fold via a `Pipeline` for Part 1,
and fit on the training split only for Part 2. Five of the six Part 2 classifiers
(Decision Tree, AdaBoost, Gradient Boosting, Random Forest, SVM) and both Part 1
classifiers (Logistic Regression, Naive Bayes) were unchanged to four decimal places;
only KNN's test accuracy shifted, from 0.9257 to 0.9286 (+0.0029). This confirms our
expectation that with very few missing values (2–6 per column out of 1400) and no
extreme outliers, the leaked statistics differ negligibly from train-only statistics.
We report the original (leaky) numbers above because they follow the assignment's
specified "preprocess, then classify" order, but the no-leakage re-run shows our
results are robust to this methodological choice.



In [27]:
# These are the results produced by our locally run, non-data-leakage version of the code.
# They are hardcoded here for reference and comparison.
leaky = {
    "LogR (CV)": logreg_acc,
    "NB (CV)": nb_acc,
    "KNN (test)": knn_test_acc,
    "DT (test)": dt_test_acc,
    "AdaBoost (test)": ada_test_acc,
    "GB (test)": gb_test_acc,
    "RF (test)": rf_test_acc,
    "SVM (test)": svm_test_acc,
}
noleak = {
    "LogR (CV)": 0.9386,
    "NB (CV)": 0.9264,
    "KNN (test)": 0.9286,
    "DT (test)": 0.9400,
    "AdaBoost (test)": 0.9429,
    "GB (test)": 0.9457,
    "RF (test)": 0.9371,
    "SVM (test)": 0.9343,
}

print("{:16s} {:>10s} {:>10s} {:>10s}".format("Model", "Leaky", "No-leak", "Diff"))
for name in leaky:
    d = noleak[name] - leaky[name]
    if abs(d) < 5e-5:  # leaky values are hardcoded at 4dp, so sub-rounding noise isn't a real diff
        d = 0.0
    print("{:16s} {:>10.4f} {:>10.4f} {:>+10.4f}".format(name, leaky[name], noleak[name], d))

Model                 Leaky    No-leak       Diff
LogR (CV)            0.9386     0.9386    +0.0000
NB (CV)              0.9264     0.9264    +0.0000
KNN (test)           0.9257     0.9286    +0.0029
DT (test)            0.9400     0.9400    +0.0000
AdaBoost (test)      0.9429     0.9429    +0.0000
GB (test)            0.9457     0.9457    +0.0000
RF (test)            0.9371     0.9371    +0.0000
SVM (test)           0.9343     0.9343    +0.0000


**Caveat 2 — Part 1 and Part 2 CV scores are not directly comparable.** Part 1's
cross-validation uses all 1400 examples, while Part 2's cross-validation is performed
only on the 1050 training examples (since 350 were held out for the test split). The
two sets of CV numbers are therefore computed on different-sized samples and should
not be compared directly against each other.

#### Overall Takeaway

Once missing values and scale are handled, the choice of classifier matters less
than expected on this problem — every model lands within two percentage points of
every other. The main practical benefit of the grid search was taming the Decision
Tree's tendency to overfit; the linear model was already competitive without any
tuning at all.

## **AI Acknowledgement**


Generative AI tools were used as follows. Conceptual questions about the assignment requirements and machine learning ideas were discussed with Google Gemini 3.8 Flash. Code review, and debugging were assisted by Anthropic Claude Fable 5.1. We reviewed all suggestions and take responsibility for the final notebook.